In [1]:
!pip install deepxde

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 12.8 MB/s eta 0:00:00


In [2]:
"""
Physics-Informed Neural Network (PINN) solver (DeepXDE + PyTorch)
for the 1D complex-valued Helmholtz equation.

Matches the LinPDE-GP script configuration exactly:
  u''(x) + k² u(x) = f,  x in [0, 1]
  u(0) = 1 + 0i,  u(1) = 0 + 0i
  rho=1, omega=2, G=2+2i  => k²=1-i
  f = 2 + 3i

Collocation note:
  The LinPDE-GP script uses only 3 interior collocation points (plus 2 boundary)
  for GP conditioning, leveraging the GP's global kernel structure to interpolate.
  PINNs lack this global structure and require denser collocation for comparable
  accuracy.  We use 998 interior points (matching a 1000-point uniform grid)
  as fixed PDE anchors so that the PINN has sufficient supervision.

Evaluation:
  x_test = np.linspace(0, 1, 1000)   — identical to LinPDE-GP
  Same metrics + same printing format as LinPDE-GP script
"""

import os
os.environ["DDE_BACKEND"] = "pytorch"

import time
import numpy as np
from typing import List, Dict, Optional

import deepxde as dde


# ============================================================
# Problem parameters (identical to LinPDE-GP script)
# ============================================================
domain = (0.0, 1.0)

# Physical parameters
rho = 1.0
omega = 2.0
G_real = 2.0
G_imag = 2.0

# Complex wavenumber squared
k_squared = rho * omega**2 / (G_real + 1j * G_imag)
k = np.sqrt(k_squared)

# Source term (constant RHS)
f_val = 2.0 + 3.0j
f_re = float(np.real(f_val))
f_im = float(np.imag(f_val))

# Dirichlet boundary conditions  (same as LinPDE-GP: g_left=1+0i, g_right=0+0i)
u_left = 1.0 + 0.0j
u_right = 0.0 + 0.0j

u_left_re = float(np.real(u_left))
u_left_im = float(np.imag(u_left))
u_right_re = float(np.real(u_right))
u_right_im = float(np.imag(u_right))

# Analytical solution (identical derivation to LinPDE-GP script)
u_particular = f_val / k_squared
B = 1.0 - u_particular
A_coeff = -(B * np.cos(k * 1.0) + u_particular) / np.sin(k * 1.0)


def analytical_solution(x):
    return A_coeff * np.sin(k * x) + B * np.cos(k * x) + u_particular


# ============================================================
# Discretization
# ============================================================
# NOTE: LinPDE-GP uses 3 interior collocation points + 2 boundary points,
#       relying on the GP kernel's global covariance structure.
#       PINNs require denser collocation; we use 998 interior points
#       (uniform grid matching 1000-point evaluation) as fixed anchors.
N_interior = 998
N_total = N_interior + 2  # 1000
x_grid = np.linspace(domain[0], domain[1], N_total).reshape(-1, 1)

x_interior = x_grid[1:-1, :]          # 998 fixed PDE anchors
x_boundary = np.array([[domain[0]],
                        [domain[1]]], dtype=np.float64)


# ============================================================
# DeepXDE / PyTorch setup
# ============================================================
dde.config.set_default_float("float64")
dde.config.set_random_seed(0)

a = float(np.real(k_squared))   #  1.0
b = float(np.imag(k_squared))   # -1.0

geom = dde.geometry.Interval(domain[0], domain[1])


# Complex Helmholtz split into coupled real system:
#   u = u_re + i u_im,   k² = a + ib,   f = f_re + i f_im
#   Re PDE:  u_re'' + a u_re - b u_im = f_re
#   Im PDE:  u_im'' + a u_im + b u_re = f_im
def pde(x, y):
    u_re = y[:, 0:1]
    u_im = y[:, 1:2]

    u_re_xx = dde.grad.hessian(y, x, component=0, i=0, j=0)
    u_im_xx = dde.grad.hessian(y, x, component=1, i=0, j=0)

    r_re = u_re_xx + a * u_re - b * u_im - f_re
    r_im = u_im_xx + a * u_im + b * u_re - f_im
    return [r_re, r_im]


# Boundary conditions (PointSetBC for exact endpoint matching)
bc_re_vals = np.array([[u_left_re], [u_right_re]], dtype=np.float64)
bc_im_vals = np.array([[u_left_im], [u_right_im]], dtype=np.float64)

bc_re = dde.icbc.PointSetBC(x_boundary, bc_re_vals, component=0)
bc_im = dde.icbc.PointSetBC(x_boundary, bc_im_vals, component=1)

# PDE data with fixed interior anchors (no random sampling)
data = dde.data.PDE(
    geom,
    pde,
    [bc_re, bc_im],
    num_domain=0,
    num_boundary=0,
    anchors=x_interior,
    num_test=0,
)

# Network: 1 input → [64, 64, 64] → 2 outputs (Re, Im)
net = dde.nn.FNN([1] + [64, 64, 64] + [2], "tanh", "Glorot normal")

model = dde.Model(data, net)

# ============================================================
# Training
# ============================================================
t_start = time.perf_counter()

model.compile("adam", lr=1e-3)
model.train(iterations=20000, display_every=5000)

model.compile("L-BFGS")
model.train(display_every=1000)

t_train = time.perf_counter() - t_start


# ============================================================
# Evaluation (same grid and protocol as LinPDE-GP script)
# ============================================================
x_test = np.linspace(0.0, 1.0, 1000)

# Predict directly on x_test (no interpolation needed — grid is identical)
t_pred_start = time.perf_counter()
y_pred = model.predict(x_test.reshape(-1, 1))  # shape (1000, 2)
t_pred = time.perf_counter() - t_pred_start

pred_re = y_pred[:, 0]
pred_im = y_pred[:, 1]

# Analytical solution on the same test grid
u_exact = analytical_solution(x_test)
true_re = np.real(u_exact)
true_im = np.imag(u_exact)


# ============================================================
# Metrics (identical function to LinPDE-GP script)
# ============================================================
def compute_metrics(u_pred: np.ndarray,
                    u_true: np.ndarray,
                    metrics: Optional[List[str]] = None) -> Dict[str, float]:
    if metrics is None:
        metrics = ["mse", "rmse", "mae", "max_error", "relative_l2", "r2"]

    u_pred = np.atleast_1d(u_pred).flatten()
    u_true = np.atleast_1d(u_true).flatten()

    metric_functions = {
        "mse": lambda yt, yp: np.mean((yt - yp) ** 2),
        "rmse": lambda yt, yp: np.sqrt(np.mean((yt - yp) ** 2)),
        "mae": lambda yt, yp: np.mean(np.abs(yt - yp)),
        "max_error": lambda yt, yp: np.max(np.abs(yt - yp)),
        "r2": lambda yt, yp: 1 - np.sum((yt - yp) ** 2) / np.sum((yt - np.mean(yt)) ** 2),
        "relative_l2": lambda yt, yp: (
            np.linalg.norm(yt - yp) / np.linalg.norm(yt)
            if np.linalg.norm(yt) > 0
            else np.inf
        ),
    }

    return {m: metric_functions[m](u_true, u_pred) for m in metrics if m in metric_functions}


# Per-component metrics
metrics_re = compute_metrics(pred_re, true_re)
metrics_im = compute_metrics(pred_im, true_im)

# Complex magnitude error (same as LinPDE-GP script)
complex_pred = pred_re + 1j * pred_im
complex_error = np.abs(complex_pred - u_exact)
metrics_complex = {
    "mse": np.mean(complex_error ** 2),
    "rmse": np.sqrt(np.mean(complex_error ** 2)),
    "mae": np.mean(complex_error),
    "max_error": np.max(complex_error),
    "relative_l2": np.linalg.norm(complex_error) / np.linalg.norm(u_exact),
    "r2": 1 - np.sum(complex_error ** 2)
         / np.sum(np.abs(u_exact - np.mean(u_exact)) ** 2),
}


# ============================================================
# Print results (same format as LinPDE-GP script)
# ============================================================
def _print_block(title, m):
    print(f"\n  --- {title} ---")
    print(f"    R² Score:            {m['r2']:12.6f}")
    print(f"    MSE:                 {m['mse']:12.6e}")
    print(f"    RMSE:                {m['rmse']:12.6e}")
    print(f"    Mean Absolute Error: {m['mae']:12.6e}")
    print(f"    Maximum Error:       {m['max_error']:12.6e}")
    print(f"    Relative L2 Error:   {m['relative_l2']:12.6e}")


print("=" * 58)
print("EVALUATION METRICS — Complex Helmholtz 1D (PINN)".center(58))
print(f"  k² = {k_squared:.4f}  (k = {k:.4f})")
print(f"  Interior collocation points: {N_interior}")
print(f"  Network: [1, 64, 64, 64, 2] tanh")
print(f"  Training time: {t_train:.2f} s")
print(f"  Prediction time: {t_pred:.4f} s")
print("=" * 58)

_print_block("Real Part", metrics_re)
_print_block("Imaginary Part", metrics_im)
_print_block("Complex Magnitude Error", metrics_complex)

print("\n" + "=" * 58)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Set the default float type to float64
Compiling model...
'compile' took 4.939779 s

Training model...

Step      Train loss                                  Test loss                                   Test metric
0         [4.14e+00, 9.12e+00, 5.00e-01, 4.79e-03]    [nan, nan, 5.00e-01, 4.79e-03]              []  

Best model at step 0:
  train loss: 1.38e+01
  test loss: nan
  test metric: []

'train' took 0.730231 s

Compiling model...
'compile' took 0.000501 s

Training model...

Step      Train loss                                  Test loss                                   Test metric
1         [3.86e+00, 8.25e+00, 5.21e-01, 3.25e-03]    [nan, nan, 5.21e-01, 3.25e-03]              []  
855       [3.65e-08, 3.95e-07, 3.59e-14, 8.28e-13]    [nan, nan, 3.59e-14, 8.28e-13]              []  

Best model at step 855:
  train loss: 4.31e-07
  test loss: nan
  test metric: []

'train' took 25.514138 s

     EVALUATION METRICS — Complex Helmholtz 1D (PINN)     
  k² = 1.0000-1.0000j  (k =